# 🧠 การวิเคราะห์ Epoch และเส้นโค้งการสรุปความรู้ทั่วไป (Generalization Curves)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Epoch Analysis and Generalization Curves**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายคำจำกัดความและความสัมพันธ์ทางคณิตศาสตร์ระหว่าง epoch, ขนาด batch และขั้นตอนการอัปเดต (update steps)
2. กำหนดฟังก์ชันทางคณิตศาสตร์ที่เป็นตัวแทนของการเรียนรู้เกิน (overfitting) โดยจำลองการลดลงเชิงทางเดียว (monotonic decay) ของการฝึกสอนเปรียบเทียบกับเส้นโค้งการตรวจสอบความถูกต้องรูปตัว U (U-shaped validation curve)
3. เขียนโค้ด **ตรวจจับการหยุดก่อนกำหนด (Early Stopping Detector) จากศูนย์** ด้วย Python ซึ่งจะเลียนแบบกลไก `patience` ในไลบรารีการเรียนรู้เชิงลึก
4. แสดงภาพ **พื้นที่การเรียนรู้เกิน (Overfitting Zone)** และ **ตัวจุดชนวนการหยุดก่อนกำหนด (Early Stopping Trigger)** บนกราฟเส้นคู่ที่แสดงค่า loss อย่างชัดเจน
5. เชื่อมโยงแนวคิดเหล่านี้กับพารามิเตอร์เริ่มต้นการฝึกสอนของ YOLO ได้แก่ `epochs=100` และ `patience=100`

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การจำลองตัววัดความสูญเสียของการเรียนรู้เกิน (Overfitting Loss Metrics)

เราจำลองค่าความสูญเสีย (loss) ทางคณิตศาสตร์ตลอด 80 epochs:
-   **Training Loss:** ลดลงแบบเอกซ์โพเนนเชียลเมื่อโมเดลปรับเข้ากับชุดข้อมูลฝึกสอน
-   **Validation Loss:** ลดลงในช่วงแรก จนกระทั่งถึงจุดต่ำสุด จากนั้นจะเริ่มเพิ่มขึ้นเมื่อโมเดลเริ่มจดจำสัญญาณรบกวนของข้อมูลฝึกสอน (การเรียนรู้เกิน หรือ overfitting)

In [ ]:
epochs = np.arange(1, 81)

# Training loss decreases monotonically
train_loss = 2.0 * np.exp(-0.08 * epochs) + 0.05 + np.random.normal(0, 0.005, len(epochs))

# Validation loss reaches a minimum around epoch 30, then rises
val_loss = 2.1 * np.exp(-0.08 * epochs) + 0.08 * np.exp(0.025 * epochs) + np.random.normal(0, 0.01, len(epochs))

## 2. การสร้าง Early Stopping ด้วย Patience

Early stopping จะเฝ้าติดตามค่า validation loss หากค่า validation loss ไม่มีการปรับปรุงดีขึ้นติดต่อกันเป็นจำนวน epoch เท่ากับ `patience` การฝึกสอนจะถูกหยุดลง และจะดึงน้ำหนักที่ดีที่สุด (best weights) จาก epoch ที่ดีที่สุดกลับคืนมา

In [ ]:
def find_early_stopping(val_losses, patience=10):
    best_loss = float('inf')
    best_epoch = 0
    no_improvement_count = 0
    
    for idx, loss in enumerate(val_losses):
        epoch = idx + 1
        if loss < best_loss:
            best_loss = loss
            best_epoch = epoch
            no_improvement_count = 0
        else:
            no_improvement_count += 1
            
        if no_improvement_count >= patience:
            return best_epoch, epoch
            
    return best_epoch, len(val_losses)

# Calculate stopping metrics for patience = 8
best_ep, stop_ep = find_early_stopping(val_loss, patience=8)
print(f"Optimal Epoch (Minimum Val Loss): {best_ep}")
print(f"Early Stopping Triggered at Epoch: {stop_ep}")

## 3. การแสดงภาพ Generalization Curves และพื้นที่การเรียนรู้เกิน (Overfitting Zone)

มาพล็อตกราฟเส้นโค้ง โดยแรเงา **Overfitting Zone** (บริเวณที่ validation loss เริ่มไต่สูงขึ้น) และทำเครื่องหมายจุดที่เกิด early stopping

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_loss, color='blue', linewidth=2, label='Training Loss')
plt.plot(epochs, val_loss, color='red', linewidth=2, label='Validation Loss')

plt.axvline(x=best_ep, color='green', linestyle='--', linewidth=2, label=f'Optimal Model (Epoch {best_ep})')
plt.axvline(x=stop_ep, color='purple', linestyle=':', linewidth=2, label=f'Stopping Trigger (Epoch {stop_ep})')

plt.axvspan(best_ep, len(epochs), color='red', alpha=0.1, label='Overfitting Zone')

plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.title('Epoch Analysis: Overfitting Detection & Early Stopping')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

ดูที่ภาพจำลองสิ:
-   **ก่อน Epoch 30 (Optimal Model):** ทั้งค่า training loss และ validation loss ลดลงทั้งคู่ โมเดลกำลังเรียนรู้ลักษณะเด่นที่เป็นแบบทั่วไป (generalized features)
-   **หลัง Epoch 30 (Overfitting Zone):** Training loss ยังคงลดลงอย่างต่อเนื่อง แต่ validation loss สูงขึ้น โมเดลกำลังจดจำรายละเอียดของชุดข้อมูลฝึกสอน (memorizing)
-   **Epoch 38 (Stopping Trigger):** ณ จุดนี้ ค่า validation loss ไม่สามารถสร้างจุดต่ำสุดใหม่ติดต่อกันเป็นเวลา 8 epochs จึงกระตุ้นเงื่อนไขการหยุดเพื่อประหยัดทรัพยากรการคำนวณและป้องกันการเรียนรู้เกินไปมากกว่านี้

## 💡 การเชื่อมโยงไปยัง YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **YOLO Early Stopping:** ภายในการฝึกสอนของ YOLO การตั้งค่าเริ่มต้นกำหนดให้ `patience=100` หากคุณตั้งค่า `epochs=300` แต่ค่า validation mAP (หรือ loss) ไม่ปรับปรุงดีขึ้นติดต่อกันเป็นเวลา 100 epochs ทาง YOLO จะหยุดการฝึกสอนโดยอัตโนมัติ และจะส่งออกน้ำหนักที่ดีที่สุดซึ่งถูกบันทึกไว้ใน epoch ที่มีการตรวจสอบความถูกต้องสูงสุด
*   สิ่งนี้ช่วยป้องกันไม่ให้เสียเวลาในการทำงานของ GPU หลายชั่วโมงไปกับโมเดลที่ความสามารถในการสรุปความรู้ทั่วไปเริ่มเสื่อมถอยลงแล้ว